# PostgreSQL Access

Write the Iris dataset to PostgreSQL and read it back.

In [ ]:
import getpass

import pandas as pd

from sklearn.datasets import load_iris

In [ ]:
host = "<server-domain>"
port = 5432
database = "<database>"
user = "<user>"

In [ ]:
df = load_iris(as_frame=True)
df.frame.head()

## SQLAlchemy 
[SQLAlchemy](https://www.sqlalchemy.org/) is a Python library that provides a database abstraction layer so you can connect to different SQL databases.
It works well with `pandas.to_sql()` and `pandas.read_sql()` and can be used with different backends.  

In [ ]:
from sqlalchemy import URL, create_engine

engine = create_engine(
    URL.create(
        "postgresql+psycopg2",
        username=user,
        password=getpass.getpass("Database password: "),
        host=host,
        port=port,
        database=database,
    )
)

df.frame.to_sql("iris_example", engine, if_exists="replace", index=False)
pd.read_sql("SELECT * FROM iris_example", engine).head()

## psycopg2 cursor

Direct `psycopg2` cursor calls are more verbose, but they show the underlying DB-API flow: open a connection, create a cursor, execute SQL, commit, fetch rows, and close the connection.

Documentation: [psycopg2](https://www.psycopg.org/docs/index.html).

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host=host,
    port=port,
    dbname=database,
    user=user,
    password=getpass.getpass("Database password: "),
)

try:
    with conn.cursor() as cursor:
        cursor.execute("DROP TABLE IF EXISTS iris_example_cursor")
        cursor.execute("""
            CREATE TABLE iris_example_cursor (
                "sepal length (cm)" DOUBLE PRECISION,
                "sepal width (cm)" DOUBLE PRECISION,
                "petal length (cm)" DOUBLE PRECISION,
                "petal width (cm)" DOUBLE PRECISION,
                target INTEGER
            )
        """)
        cursor.executemany("""
            INSERT INTO iris_example_cursor (
                "sepal length (cm)",
                "sepal width (cm)",
                "petal length (cm)",
                "petal width (cm)",
                target
            )
            VALUES (%s, %s, %s, %s, %s)
        """, df.frame.itertuples(index=False, name=None))
        conn.commit()

        cursor.execute("SELECT * FROM iris_example_cursor LIMIT 5")
        rows = cursor.fetchall()
        columns = [column.name for column in cursor.description]

    display(pd.DataFrame(rows, columns=columns))
finally:
    conn.close()